In [ ]:
# !pip install -q langchain-community pypdf langchain-google-genai google-genai chromadb
# !pip install -q langchain-text-splitters
# !pip install -q langchain-chroma

In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from google import genai
from google.genai.errors import ServerError
from langchain_community.document_loaders import PyPDFLoader
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

C:\Users\Admin\AppData\Local\Temp\ipykernel_9704\2664611870.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# 1. Set your API Key
# os.environ["GEMINI_API_KEY"] = ""
# client = genai.Client()


In [2]:
loader = PyPDFLoader('attention.pdf')
docs = loader.load()
# print(docs)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)
documents = text_splitter.split_documents(docs)

# Corrected variable name slice (documents instead of document)
print(documents[:1])

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszka

In [4]:
from langchain_chroma import Chroma
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize your Chroma vector store with the correct current embedding model name
db = Chroma.from_documents(
    documents=documents, 
    embedding=GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        # google_api_key=""
    ),
    persist_directory="./chroma_db"
)

print("Vector database created successfully!")

Vector database created successfully!


In [8]:
query = "who are the authors of attention is all you need?"
import json
# retrived_result = db.similarity_search(query)
results = db.similarity_search_with_score(query, k=1)
# print(retrived_result[0].page_content)
# Format the results into a Python list of dictionaries
structured_data = []
for doc, score in results:
    structured_data.append({
        "content": doc.page_content,
        "metadata": doc.metadata,
        "score": float(score)
    })

# Convert to a clean JSON string and output it
json_output = json.dumps(structured_data, indent=4)
print(json_output)

[
    {
        "content": "Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani\u2217\nGoogle Brain\navaswani@google.com\nNoam Shazeer\u2217\nGoogle Brain\nnoam@google.com\nNiki Parmar\u2217\nGoogle Research\nnikip@google.com\nJakob Uszkoreit\u2217\nGoogle Research\nusz@google.com\nLlion Jones\u2217\nGoogle Research\nllion@google.com\nAidan N. Gomez\u2217\u2020\nUniversity of Toronto\naidan@cs.toronto.edu\n\u0141ukasz Kaiser\u2217\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin\u2217\u2021\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, 

In [ ]:

# # 2. List of models to cycle through if a 503 high-demand spike occurs
# models_to_try = ["gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.8-flash"]
# response_text = None

# # 3. Automatic fallback loop to bypass server congestion
# for model_name in models_to_try:
#     try:
#         response = client.models.generate_content(
#             model=model_name,
#             contents="Hello, Gemini!"
#         )
#         response_text = response.text
#         break  # Stop looping once a successful response is retrieved
#     except ServerError:
#         continue  # Instantly try the next model if 503 error hits

# # 4. Strictly output the final response value
# if response_text:
#     print(response_text)
# else:
#     print("All models are currently experiencing high server traffic. Please re-run this cell in a few seconds.")